In [ ]:
from pathlib import Path

import h5py
import numpy as np
import polars as pl

import gambit as pq


def create_data(source: Path, destination: Path) -> pl.DataFrame:
    """Create the compact example options dataset from a local source archive."""
    frames: list[pl.DataFrame] = []

    with h5py.File(source, "r") as archive:
        dates = np.arange(np.datetime64("2023-01-01"), np.datetime64("2023-01-10"))
        for date in dates:
            date_text = np.datetime_as_string(date, unit="D").replace("-", "_")
            key = f"D_{date_text}"
            if key not in archive:
                continue

            trades = (
                pq.hdf5_to_df(source, key)
                .with_columns(
                    pl.date("okey_yr", "okey_mn", "okey_dy").alias("expiry"),
                    ((pl.col("uBid") + pl.col("uAsk")) * 0.5).alias("umid"),
                )
                .select(
                    "timestamp",
                    pl.col("okey_cp").alias("put_call"),
                    "expiry",
                    pl.col("okey_xx").alias("strike"),
                    pl.col("prtPrice").alias("price"),
                    pl.col("prtVolume").alias("volume"),
                    pl.col("prtIv").alias("iv"),
                    pl.col("prtDe").alias("delta"),
                    "umid",
                )
                .with_columns(
                    pl.concat_str(
                        pl.col("put_call").str.slice(0, 1),
                        pl.col("strike").cast(pl.Int64).cast(pl.String),
                        pl.col("expiry").cast(pl.String),
                        separator="-",
                    ).alias("symbol"),
                    pl.col("timestamp").cast(pl.Datetime("ns")),
                )
                .sort(["symbol", "timestamp"])
            )
            bars = (
                trades.group_by_dynamic("timestamp", every="5m", group_by="symbol")
                .agg(
                    pl.col("price").first().alias("o"),
                    pl.col("price").max().alias("h"),
                    pl.col("price").min().alias("l"),
                    pl.col("price").last().alias("c"),
                    pl.col("volume").sum().alias("v"),
                    pl.col("umid").last(),
                    pl.col("iv").last(),
                    pl.col("delta").last(),
                )
                .filter(pl.col("c").is_finite())
                .select("timestamp", "symbol", "o", "h", "l", "c", "v", "umid", "iv", "delta")
            )
            frames.append(bars)

    result = pl.concat(frames) if frames else pl.DataFrame()
    result.write_csv(destination)
    return result
